In [7]:
# =====================================================================
# CELL 1: IMPORT & LOAD UNIFIED MASTER DATA
# =====================================================================
import pandas as pd
import numpy as np
import random

from warehouse_env import warehouse_positions, io_point

# 1. Load the unified master file (contains both abc_class and K-Means Cluster)
df_master = pd.read_csv('../../features/sku_clusters_final.csv')

# 2. Verify the loaded data
print(f"I/O Point: {io_point}")
print(f"Total warehouse slots available: {len(warehouse_positions)}")
print(f"Total SKUs to slot (Master Data): {len(df_master)}")

I/O Point: (0, 0)
Total warehouse slots available: 4250
Total SKUs to slot (Master Data): 4130


In [8]:
#CALCULATE ABC CLASSIFICATION AND VELOCITY (PIPELINE VERIFICATION)

# 1. Load the cleaned data from the 'data' directory
# Note: Adjust the relative path '../../data/' based on your actual structure
df = pd.read_csv('../../data/processed/data_clean.csv')

# 2. Calculate Velocity (Total Quantity) and Pick Frequency per SKU
df_abc = df.groupby('stock_code').agg(
    total_quantity=('quantity', 'sum'),
    pick_frequency=('invoice_no', 'nunique'),
    total_revenue=('quantity', lambda x: (x * df.loc[x.index, 'price']).sum())
).reset_index()

# 3. Sort by total quantity (Velocity) in descending order
df_abc = df_abc.sort_values(by='total_quantity', ascending=False)

# 4. Calculate cumulative percentage for Pareto principle
df_abc['cum_percent'] = df_abc['total_quantity'].cumsum() / df_abc['total_quantity'].sum()

# 5. Classify SKUs into A (Top 80%), B (Next 15%), C (Bottom 5%)
conditions = [
    (df_abc['cum_percent'] <= 0.80),
    (df_abc['cum_percent'] > 0.80) & (df_abc['cum_percent'] <= 0.95),
    (df_abc['cum_percent'] > 0.95)
]
choices = ['A', 'B', 'C']
df_abc['ABC_Class'] = np.select(conditions, choices, default='C')

# Display the verification results
print("--- ABC CLASSIFICATION STATISTICS ---")
print(df_abc['ABC_Class'].value_counts())
display(df_abc.head())

--- ABC CLASSIFICATION STATISTICS ---
ABC_Class
C    2378
B     995
A     878
Name: count, dtype: int64


,stock_code,total_quantity,pick_frequency,total_revenue,cum_percent,ABC_Class
612,21212,60141,1853,31738.10,0.010312,A
3641,85123A,58487,3281,158590.87,0.020341,A
2894,84077,55091,501,11418.05,0.029788,A
3613,85099B,49875,1966,89114.78,0.038340,A
113,17003,48374,230,8922.80,0.046634,A


In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

# =====================================================================
# 1. OPTIMIZED PREPROCESSING PIPELINE FUNCTION
# =====================================================================
def optimize_preprocessing_pipeline(data, feature_cols, lower_q=0.0, upper_q=0.995):
    """
    Executes a robust 4-step preprocessing sequence: 
    Raw Data -> Winsorization -> Log1p Transformation -> RobustScaler
    """
    df_out = data[feature_cols].copy()
    
    for col in feature_cols:
        u_limit = df_out[col].quantile(upper_q)
        df_out[col] = np.where(df_out[col] > u_limit, u_limit, df_out[col])
        df_out[col] = np.log1p(df_out[col])
        
    scaler = RobustScaler()
    scaled_data = scaler.fit_transform(df_out)
    
    df_scaled = pd.DataFrame(scaled_data, columns=feature_cols, index=data.index)
    return df_scaled, scaler

# =====================================================================
# 2. LOAD THE RAW DATA FOR PREPROCESSING & CLUSTERING
# =====================================================================
# Load the aggregated master dataset 
df_raw = pd.read_csv('../../features/sku_features.csv')

# Define the exact feature columns
features = ['monthly_velocity', 'total_sales', 'pick_frequency']

print("--- STARTING OPTIMIZED PREPROCESSING PIPELINE ---")
df_scaled, trained_scaler = optimize_preprocessing_pipeline(df_raw, features)
print("[v] Preprocessing sequence completed: Winsorization -> Log1p -> RobustScaler.")

# =====================================================================
# 3. K-MEANS CLUSTERING EXECUTION
# =====================================================================
N_CLUSTERS = 3
kmeans_model = KMeans(n_clusters=N_CLUSTERS, init='k-means++', random_state=42, n_init=10)

# Assign labels directly back to the master dataframe
df_raw['Cluster'] = kmeans_model.fit_predict(df_scaled)
print(f"[v] Successfully grouped {len(df_raw):,} unique SKUs into {N_CLUSTERS} K-Means clusters.")
print("="*70)

# =====================================================================
# 4. EXPORT CLUSTERING RESULTS TO FILE (ESSENTIAL COLUMNS ONLY)
# =====================================================================
# Keep only the crucial columns needed for the slotting simulation
final_cols = [
    'stock_code', 
    'abc_class', 
    'Cluster', 
    'pick_frequency', 
    'monthly_velocity', 
    'total_sales'
]
df_export = df_raw[final_cols]

output_path = '../../features/sku_clusters_final.csv'
df_export.to_csv(output_path, index=False)

print(f"--- SUCCESS: Cleaned master results exported to {output_path} ---")
print(df_export.head())
print("="*70)

--- STARTING OPTIMIZED PREPROCESSING PIPELINE ---
[v] Preprocessing sequence completed: Winsorization -> Log1p -> RobustScaler.
[v] Successfully grouped 4,130 unique SKUs into 3 K-Means clusters.
--- SUCCESS: Cleaned master results exported to ../../features/sku_clusters_final.csv ---
  stock_code abc_class  Cluster  pick_frequency  monthly_velocity  total_sales
0     85099B         A        1            1990       3868.769231     91226.42
1      20725         A        1            1539       1645.307692     36250.84
2      21931         A        1            1179       1219.153846     31515.16
3      22383         A        1            1149       1075.750000     21390.12
4      20727         A        1            1115       1145.076923     24001.28
